In [47]:
import os
from anthropic import Anthropic
import json
from pathlib import Path


## Session 1

In [6]:
question = "Where can I get ice-cream in Hanover, New Hampshire within walking distance of Dartmouth College?"
client = Anthropic(
    api_key = os.environ.get("ANTHROPIC_API_KEY")
)

messages = [{"role": "user", "content": question}]

tool_request = client.messages.create(
    model = "claude-haiku-4-5-20251001",
    max_tokens = 1024,
    messages = messages
)

print(tool_request.content)
print(tool_request.usage.output_tokens)

messages.append({"role": "assistant", "content": tool_request.content
})

question2 = "Following up from my previous question: tell me more about the Ben and Jerry's in Hanover, NH. Where exactly is it?"
messages.append({"role": "user", "content": question2})

tool_request = client.messages.create(
    model = "claude-haiku-4-5-20251001",
    max_tokens = 1024,
    messages = messages
)

print(tool_request.content)
print(tool_request.usage.output_tokens)



[TextBlock(citations=None, text="# Ice Cream Near Dartmouth College in Hanover\n\nHere are some options within walking distance:\n\n## Main Green Area\n- **Ben & Jerry's** - Located on or near Main Street in downtown Hanover, this is typically the most accessible option for students\n\n## Downtown Hanover\nThe downtown area (a short walk from campus) has several options along Main Street and the surrounding blocks, though specific shops may change.\n\n## On Campus\n- Dartmouth dining facilities occasionally serve ice cream, so check what's available in your residential area or dining hall\n\n## Tips\n- Main Street is the hub for shops and restaurants, generally a 5-15 minute walk from most dorms\n- The Hanover area is very walkable, so most downtown establishments are accessible on foot\n- I'd recommend checking current listings or asking at your dorm, as businesses do open and close\n\nSince specific businesses can change seasonally or relocate, I'd suggest:\n- Checking Google Maps fo

In [17]:
client = Anthropic(
    api_key = os.environ.get("ANTHROPIC_API_KEY")
)

instruction = "Extract the key information from this job description: Want to build AI that reaches a global bank’s real users, but not sure where to start? UBS is putting AI at the core of how we work, and the Chief Transformation Office is where that transformation is engineered. We’re looking for ambitious students to join us for a 2027 summer internship as an AI intern. As An AI Intern, You’ll Help Us Build, Test, And Ship AI-enabled Applications That Improve How Thousands Of Colleagues Work Across The Firm. Working Alongside Experienced Engineers And Product Managers, You’ll design and build AI agents that automate, augment, or fully reinvent how work gets done across UBS apply core generative AI techniques, including prompt engineering, retrieval augmented generation (RAG), embeddings, MCP, LangGraph, and emerging agentic patterns, to deliver real production cases map and redesign business workflows around AI agents, pinpointing where automation and reasoning remove friction and improve outcomes build across the agent spectrum, using no-code and low-code platforms for fast iteration, and pro-code frameworks where deeper customization, integrations, or extensions are needed iterate on agent quality, improving accuracy, reliability, and usefulness through systematic testing, evaluation, and refinement collaborate with product, business, and platform teams to turn high-impact use cases into deployed agents that real users rely on" 
messages = [{"role": "user", "content": instruction}]

response = client.messages.create(
    model = "claude-haiku-4-5-20251001",
    max_tokens = 1024,
    messages = messages,
    output_config = {
        "format": {
            "type": "json_schema",
            "schema": {
                "type": "object",
                "properties": {
                    "company name": {"type": "string"},
                    "role/title": {"type": "string"},
                    "level": {"type": "string"},
                    "skills": {"type": "array"},
                    "deadline": {"type": "string"}
                },
                "required": ["company name", "role/title", "level", "skills"],
                "additionalProperties": False,
            },
        }
    },
)

textBlock = ""
for block in response.content:
    if block.type == "text":
        textBlock = block

record = json.loads(block.text)

print(record)

# checking to see if I can iterate through the skills block
for skill in record["skills"]:
    print(skill)

{'company name': 'UBS', 'role/title': 'AI Intern', 'level': 'Internship', 'skills': ['Generative AI', 'Prompt Engineering', 'Retrieval Augmented Generation (RAG)', 'Embeddings', 'MCP', 'LangGraph', 'Agentic Patterns', 'AI Agent Design', 'Product Management', 'Workflow Redesign', 'No-code Platforms', 'Low-code Platforms', 'Pro-code Frameworks', 'Testing', 'Evaluation', 'Refinement'], 'deadline': '2027 Summer'}
Generative AI
Prompt Engineering
Retrieval Augmented Generation (RAG)
Embeddings
MCP
LangGraph
Agentic Patterns
AI Agent Design
Product Management
Workflow Redesign
No-code Platforms
Low-code Platforms
Pro-code Frameworks
Testing
Evaluation
Refinement


## Session 3

In [50]:
def getJobData(keyword: str):
    keyword = keyword.lower() # normalize the keyword
    folder = Path("postings") # get filepath to the postings
    results = {}

    for path in folder.glob("*.txt"):
        results[path.name] = []
        text = path.read_text(encoding="utf-8").lower()
        i = 0
        end = len(text)
        # find all instances of the keyword
        while i != -1:
            pos = text.find(keyword, i, end)
            if pos != -1:
                if pos >= 60:
                    results[path.name].append(text[pos-60:pos+61])
                else:
                    results[path.name].append(text[0:pos+61])
                i = pos + len(keyword)
            else: i = -1
    return results

tools = [
    {
        "name": "get_jd_info",
        "description": "Search all of the user's saved job descriptions to get information about a particular skill or qualification needed for them. Ex: Needing to know PyTorch, Kubernetes, C++, etc. Using this tool will return a dictionary where the key is the filename and the value is a list of snippets containing the keyword from the JD. Note that 1 list corresponds to 1 job description, and a list can have 0, 1 or more snippets in it.  If a key has an empty list, this indicates that the keyword was not in the JD.",
        "input_schema": {
            "type": "object",
            "properties": {
                "keyword": {
                    "type": "string",
                    "description": "The skill/qualification you want to learn more about. Note, this tool only returns things based on keyword search, so you'll have to be specific about what you want. For instance, you cannot generalize and ask about AI-based skills and expect to get back an answer that talks about knowing LangGraph, how to build harnesses, etc. This is one case insensitive keyword/phrase per call. You'll get back matches or no matches with the keyword \"AI skills\". Example input for tool use: PyTorch. ",
                },
            },
            "required": ["keyword"],
        },
    }
]

client = Anthropic(
    api_key = os.environ.get("ANTHROPIC_API_KEY")
)

# 1st Response
messages = [{"role": "user", "content": "I have a bunch of job descriptions saved and I need to know which of them require/recommend that I know C++."}]
result = client.messages.create(
    model = "claude-haiku-4-5-20251001",
    max_tokens = 1024,
    messages = messages,
    tools = tools,
    tool_choice = {"type": "auto", "disable_parallel_tool_use": True}
)

print("Response 1:", result)

tool_input = None
tool_block = None

while result.stop_reason == "tool_use":
    tool_block = None
    tool_input = None
    for block in result.content:
        if block.type == "tool_use":
            tool_block = block
            if tool_block.name == "get_jd_info":
                tool_input = block.input["keyword"]

    if tool_input == None:
        print("Claude requested an unknown tool: ", tool_block.name)
        print("Notifying Claude so it can re-request with the correct tool...")

        # add Claude's message to the list
        messages.append({"role": "assistant", "content": result.content})

        # add tool response to the list
        messages.append({"role":"user",
                            "content": [{
                                "type": "tool_result",
                                "tool_use_id": tool_block.id,
                                "content": "Unknown tool was called. Look at the available tools again carefully to make sure you're calling the correct tool!"
                            }]
                        })

    else:
        # call tool
        job_info = getJobData(tool_input)

        # need to convert tool result content to a string or list
        job_info = str(job_info)

        # add Claude's message to the list
        messages.append({"role": "assistant", "content": result.content})

        # add tool response to the list
        messages.append({"role":"user",
                            "content": [{
                                "type": "tool_result",
                                "tool_use_id": tool_block.id,
                                "content": job_info
                            }]
                        })

    result = client.messages.create(
        model = "claude-haiku-4-5-20251001",
        max_tokens = 1024,
        messages = messages,
        tools = tools,
        tool_choice = {"type": "auto", "disable_parallel_tool_use": True}
    )

final_ans = ""
for block in result.content:
    if block.type == "text":
        final_ans = block

print("Claude's final answer:", final_ans.text)


Response 1: Message(id='msg_011CfSamzADP2WXsZ5oVZBvt', container=None, content=[ToolUseBlock(id='toolu_01NWfP9ZH77FoW8v8R6FSLQL', caller=DirectCaller(type='direct'), input={'keyword': 'C++'}, name='get_jd_info', type='tool_use')], model='claude-haiku-4-5-20251001', role='assistant', stop_details=None, stop_reason='tool_use', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=610, output_tokens=49, output_tokens_details=None, server_tool_use=None, service_tier='standard'), diagnostics=None)
Claude's final answer: Based on your saved job descriptions, **3 positions require or recommend C++** knowledge:

1. **PIMCO Tech Analyst, SWE** - Lists C++ as one of the strong programming skills needed (along with Python, Java, C#, SQL, RDBMS, and data warehouse skills)

2. **Qualcomm Silicon Validation Internship*